In [ ]:
!pip install konlpy
!apt update
!apt install -y openjdk-17-jdk
!pip install pandas

In [ ]:
!pip uninstall -y numpy scipy scikit-learn
!pip install numpy==1.26.4
!pip install scipy==1.11.4
!pip install scikit-learn==1.3.2

In [ ]:
import re
import pandas as pd
import torch
import torch.nn as nn

from konlpy.tag import Okt
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# =========================
# 데이터 로드
# =========================
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)

df['target'] = df['rating'].apply(lambda x : 1 if x > 0.5 else 0)

df['clean'] = df['review'].apply(
    lambda x : re.sub(r'[^가-힣\s]', '', str(x))
)

# =========================
# 토크나이저
# =========================
okt = Okt()

def kor_tokenizer(text):
    return [
        word for word, pos in okt.pos(text, stem=True)
        if pos in ['Noun', 'Verb', 'Adjective']
        and len(word) >= 2
    ]

# =========================
# 단어사전 구축
# =========================
vocab = Counter()

for text in df['clean']:
    vocab.update(kor_tokenizer(text))

vocab_size = 10000

word_to_index = {
    word: idx + 2
    for idx, (word, count)
    in enumerate(vocab.most_common(vocab_size))
}

word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

index_to_word = {
    idx: word
    for word, idx in word_to_index.items()
}

# =========================
# 문장 -> 시퀀스
# =========================
def word2Sequence(text):
    return [
        word_to_index.get(word, 1)
        for word in kor_tokenizer(text)
    ]

# =========================
# Padding
# =========================
MAX_LEN = 500

def padding(texts):

    result = []

    for text in texts:

        seq = word2Sequence(text)

        # 최대 길이 제한
        seq = seq[:MAX_LEN]

        # tensor 변환
        seq = torch.LongTensor(seq)

        result.append(seq)

    return pad_sequence(
        result,
        batch_first=True,
        padding_value=0
    )

X = padding(df['clean'])

y = torch.FloatTensor(df['target'].values)

print(X.shape)
print(y.shape)

# =========================
# train test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================
# DataLoader
# =========================
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)

# =========================
# Device
# =========================
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(device)

# =========================
# RNN 모델
# =========================
class RNNClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        # B L -> B L D
        x = self.embedding(x)

        # output : B L H
        # hidden : 1 B H
        output, hidden = self.rnn(x)

        # 마지막 hidden
        hidden = hidden.squeeze(0)

        logits = self.fc(hidden)

        return logits

# =========================
# 모델 생성
# =========================
model = RNNClassifier(
    vocab_size=vocab_size + 2,
    embedding_dim=128,
    hidden_dim=64
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# =========================
# 학습
# =========================
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch).squeeze(1)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f'Epoch {epoch+1} Loss : {total_loss/len(train_loader):.4f}'
    )

# =========================
# 평가
# =========================
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch).squeeze(1)

        pred = torch.sigmoid(logits) >= 0.5

        correct += (pred == y_batch).sum().item()

        total += y_batch.size(0)

print(f'Accuracy : {correct/total:.4f}')

# =========================
# 예측 함수
# =========================
def predict_sentiment(text):

    model.eval()

    seq = word2Sequence(text)

    seq = seq[:MAX_LEN]

    seq = torch.LongTensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():

        logits = model(seq)

        prob = torch.sigmoid(logits)

    return prob.item()

# =========================
# 테스트
# =========================
text = "진짜 너무 재미있고 감동적인 영화였다"

score = predict_sentiment(text)

print(score)

if score >= 0.5:
    print("긍정")
else:
    print("부정")

In [ ]:
# import pandas as pd
# # https://github.com/skn29teacher/skn29_lecture/blob/main/data_nlp/daum_movie_review.csv
# url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
# pd.read_csv(url)